# ProcessPath_AI — Final Report
## BPI Challenge 2020: TU/e Travel Permit Process

---

| | |
|---|---|
| **Dataset** | `PermitLog.xes` — 33 MB, BPI Challenge 2020 |
| **Organisation** | Eindhoven University of Technology (TU/e) |
| **Period covered** | October 2016 – December 2018 |
| **Notebooks** | 01 Exploration · 02 Structure · 03 Discovery · 04 Bottlenecks · 05 Conformance · 06–08 Predictive |
| **Toolchain** | PM4Py 2.7 · XGBoost 3.3 · SHAP 0.52 · Python 3.13 |

---

## Executive Summary

This report presents a complete process mining analysis of TU/e's travel permit reimbursement process.
7,065 permit cases were analysed across 18 months. The median case takes **71.7 days** from permit
submission to payment, with the top-tertile threshold at **101 days**.

Five validated findings drive five prioritised recommendations:

| Priority | Finding | Recommendation |
|---|---|---|
| 🔴 1 | 991 cases (14%) are permanently stuck on *Send Reminder* | Automated escalation after 30 days |
| 🔴 2 | 17.1% of cases have travel-before-approval violations | Hard system gate: no booking without permit |
| 🟡 3 | Scheduling delay = 69% of case duration | Reclassify — not a process problem |
| 🟡 4 | Early warning model: AUC 0.967 at k=8 events | Deploy with quarterly retraining |
| 🟢 5 | Data drift detected: cases process faster year-on-year | Monitor `elapsed_days` distribution monthly |

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

T = Path('../outputs/tables')
F = Path('../outputs/figures')
OUT = Path('../outputs/figures')

# ── Load all pre-computed results ────────────────────────────────
durations    = pd.read_csv(T / 'case_durations.csv')
bottleneck_w = pd.read_csv(T / 'bottleneck_waiting_time.csv')
bottleneck_c = pd.read_csv(T / 'bottleneck_combined.csv')
stuck_vs     = pd.read_csv(T / 'stuck_vs_resolved_duration.csv')
conformance  = pd.read_csv(T / 'conformance_replay.csv')
viol_dept    = pd.read_csv(T / 'violation_by_department.csv')
emp_wait     = pd.read_csv(T / 'employee_wait_split.csv')
pred_models  = pd.read_csv(T / 'predictive_model_summary.csv')
prefix_auc   = pd.read_csv(T / 'prefix_auc_results.csv')
temporal_cv  = pd.read_csv(T / 'temporal_cv_results.csv')
kfold_vs_t   = pd.read_csv(T / 'temporal_vs_kfold.csv')
activity_freq= pd.read_csv(T / 'activity_frequency.csv')
feat_drift   = pd.read_csv(T / 'feature_drift_by_quarter.csv', index_col=0)
train_size   = pd.read_csv(T / 'training_size_sensitivity.csv')

print('All result files loaded.')

All result files loaded.


---
## 1. Dataset at a Glance

In [2]:
n_cases    = len(durations)
n_events   = 86581
n_acts     = 51
n_variants = 1478
med_dur    = durations['duration_days'].median()
mean_dur   = durations['duration_days'].mean()
p67_dur    = durations['duration_days'].quantile(2/3)
p33_dur    = durations['duration_days'].quantile(1/3)
fit_mean   = conformance['fitness'].mean()
fit_perfect= (conformance['fitness'] == 1.0).mean()

stats = [
    ('Cases',                    f'{n_cases:,}'),
    ('Events',                   f'{n_events:,}'),
    ('Activities',               f'{n_acts}'),
    ('Variants',                 f'{n_variants:,}'),
    ('Median duration',          f'{med_dur:.1f} days'),
    ('Mean duration',            f'{mean_dur:.1f} days'),
    ('P33 (short/med boundary)', f'{p33_dur:.1f} days'),
    ('P67 (med/long boundary)',  f'{p67_dur:.1f} days'),
    ('Mean trace fitness',       f'{fit_mean:.3f}'),
    ('Perfect-fit cases',        f'{fit_perfect:.1%}'),
]
df_stats = pd.DataFrame(stats, columns=['Metric','Value'])
print(df_stats.to_string(index=False))

                  Metric      Value
                   Cases      7,065
                  Events     86,581
              Activities         51
                Variants      1,478
         Median duration  71.7 days
           Mean duration  87.4 days
P33 (short/med boundary)  50.4 days
 P67 (med/long boundary) 101.1 days
      Mean trace fitness      0.960
       Perfect-fit cases      55.1%


---
## 2. Summary Dashboard

In [3]:
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

# ── Panel A: Case duration distribution ────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
dur  = durations['duration_days'].clip(upper=400)
ax_a.hist(dur, bins=60, color='steelblue', alpha=0.75, edgecolor='none')
ax_a.axvline(med_dur, color='#1a9641', linewidth=2, linestyle='--', label=f'Median {med_dur:.0f}d')
ax_a.axvline(p33_dur, color='#fdae61', linewidth=1.5, linestyle=':', label=f'P33 {p33_dur:.0f}d')
ax_a.axvline(p67_dur, color='#d7191c', linewidth=1.5, linestyle=':', label=f'P67 {p67_dur:.0f}d')
ax_a.set_xlabel('Case duration (days, clipped at 400)')
ax_a.set_ylabel('Cases')
ax_a.set_title('A. Case Duration Distribution', fontweight='bold')
ax_a.legend(fontsize=8)

# ── Panel B: Top 8 bottlenecks by waiting time ─────────────────
ax_b = fig.add_subplot(gs[0, 1])
bottleneck_w['median_wait_d'] = bottleneck_w['median_wait_h'] / 24
top_wait = bottleneck_w.nlargest(8, 'median_wait_d')
labels   = [a.replace(' by ', '\nby ')[:35] for a in top_wait['concept:name']]
ax_b.barh(labels[::-1], top_wait['median_wait_d'][::-1], color='#d7191c', alpha=0.85)
ax_b.set_xlabel('Median waiting time (days)')
ax_b.set_title('B. Top 8 Bottlenecks — Waiting Time', fontweight='bold')
ax_b.tick_params(axis='y', labelsize=7)

# ── Panel C: Compliance breakdown ──────────────────────────────
ax_c = fig.add_subplot(gs[1, 0])
n_type_a   = 746
n_type_b   = 583
n_compliant= 5736
n_no_trip  = n_cases - n_type_a - n_type_b - n_compliant
wedge_sizes = [n_compliant, n_type_b, n_type_a, n_no_trip]
wedge_labels= [f'Compliant\n{n_compliant:,} ({n_compliant/n_cases:.1%})',
               f'Type B\n{n_type_b} ({n_type_b/n_cases:.1%})',
               f'Type A\n{n_type_a} ({n_type_a/n_cases:.1%})',
               f'No trip\n{n_no_trip}']
colors_pie = ['#1a9641','#fdae61','#d7191c','#aaaaaa']
ax_c.pie(wedge_sizes, labels=wedge_labels, colors=colors_pie,
         startangle=90, textprops={'fontsize': 8},
         wedgeprops={'edgecolor':'white','linewidth':1.5})
ax_c.set_title('C. Travel Ordering Compliance', fontweight='bold')

# ── Panel D: Temporal CV AUC by prefix ─────────────────────────
ax_d = fig.add_subplot(gs[1, 1])
num_rows = kfold_vs_t.copy()
ax_d.bar(num_rows['k'] - 0.2, num_rows['kfold_auc_mean'], 0.38,
         label='Standard k-fold CV', color='#2c7bb6', alpha=0.85)
ax_d.bar(num_rows['k'] + 0.2, num_rows['temporal_auc_mean'], 0.38,
         label='Temporal CV (deployable)', color='#d7191c', alpha=0.85)
ax_d.set_xticks(num_rows['k'])
ax_d.set_xticklabels([f'k={k}' for k in num_rows['k']])
ax_d.set_ylabel('ROC-AUC')
ax_d.set_ylim(0.5, 1.05)
ax_d.set_title('D. Early Warning AUC: k-fold vs Temporal CV', fontweight='bold')
ax_d.legend(fontsize=8)
ax_d.grid(axis='y', alpha=0.3)

# ── Panel E: SHAP top features (complete model) ────────────────
ax_e = fig.add_subplot(gs[2, 0])
feat_imp = pd.read_csv(T / 'predictive_feature_importance.csv', index_col=0)
top8 = feat_imp['XGBoost'].sort_values(ascending=False).head(8)
clean = [f.replace('_', ' ').replace('case:', '') for f in top8.index]
ax_e.barh(clean[::-1], top8.values[::-1], color='#d7191c', alpha=0.85)
ax_e.set_xlabel('Importance (% of total)')
ax_e.set_title('E. Top Features — XGBoost (complete model)', fontweight='bold')
ax_e.tick_params(axis='y', labelsize=8)

# ── Panel F: Stuck vs resolved case durations ──────────────────
ax_f = fig.add_subplot(gs[2, 1])
med_stuck    = stuck_vs.loc[stuck_vs['Unnamed: 0'].str.startswith('Stuck'), '50%'].values[0]
med_resolved = stuck_vs.loc[stuck_vs['Unnamed: 0'] == 'Resolved', '50%'].values[0]
groups_f   = ['Stuck\n(ends on\nSend Reminder)', 'Resolved\n(reaches\nPayment)']
colors_f   = ['#d7191c', '#1a9641']
bars = ax_f.bar(groups_f, [med_stuck, med_resolved], color=colors_f, alpha=0.85, width=0.5)
for bar, v in zip(bars, [med_stuck, med_resolved]):
    ax_f.text(bar.get_x() + bar.get_width()/2, v + 2,
              f'{v:.0f}d', ha='center', fontsize=11, fontweight='bold')
ax_f.set_ylabel('Median duration (days)')
ax_f.set_title('F. Stuck vs Resolved: Median Duration\n(991 stuck cases = 14% of all)', fontweight='bold')
ax_f.set_ylim(0, 165)
ax_f.grid(axis='y', alpha=0.3)

fig.suptitle('ProcessPath_AI — Summary Dashboard\n'
             'BPI Challenge 2020 · TU/e Travel Permits · 7,065 cases · 18 months',
             fontsize=14, fontweight='bold', y=0.98)

fig.savefig(OUT / 'report_dashboard.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved report_dashboard.png')

Saved report_dashboard.png


---
## 3. Finding 1 — The Send Reminder Bottleneck

In [4]:
n_stuck = 991
med_stuck_v    = stuck_vs.loc[stuck_vs['Unnamed: 0'].str.startswith('Stuck'), '50%'].values[0]
med_resolved_v = stuck_vs.loc[stuck_vs['Unnamed: 0'] == 'Resolved', '50%'].values[0]
excess_days    = med_stuck_v - med_resolved_v
pct_stuck      = n_stuck / n_cases

# Convert hours → days
bottleneck_w['median_wait_d'] = bottleneck_w['median_wait_h'] / 24
bottleneck_w['p95_wait_d']    = bottleneck_w['p95_wait_h']    / 24

print('=== FINDING 1: Send Reminder Bottleneck ===')
print(f'Stuck cases (last event = Send Reminder): {n_stuck} ({pct_stuck:.1%} of all cases)')
print(f'Median duration — stuck: {med_stuck_v:.0f}d  |  resolved: {med_resolved_v:.0f}d')
print(f'Excess delay per stuck case: {excess_days:.0f} days')
print()
top5_wait = bottleneck_w.nlargest(5, 'median_wait_d')[['concept:name','median_wait_d','p95_wait_d']]
print('Top 5 activities by median waiting time:')
print(top5_wait.round(1).to_string(index=False))
print()
sr = bottleneck_w[bottleneck_w['concept:name'] == 'Send Reminder']
if len(sr):
    print(f'Send Reminder — median wait: {sr["median_wait_d"].values[0]:.1f}d  '
          f'(p95: {sr["p95_wait_d"].values[0]:.1f}d)')
print()
print('RECOMMENDATION: Automated escalation to manager after 30 days on Send Reminder.')
print(f'Expected impact: eliminate ~{excess_days:.0f}d excess for {n_stuck} cases.')

=== FINDING 1: Send Reminder Bottleneck ===
Stuck cases (last event = Send Reminder): 991 (14.0% of all cases)
Median duration — stuck: 134d  |  resolved: 63d
Excess delay per stuck case: 71 days

Top 5 activities by median waiting time:
               concept:name  median_wait_d  p95_wait_d
              Send Reminder           52.8       140.5
   Permit SAVED by EMPLOYEE           21.2        55.3
                 Start trip           19.4       103.4
 Permit REJECTED by MISSING           14.7       120.4
Permit REJECTED by DIRECTOR           12.5        19.3

Send Reminder — median wait: 52.8d  (p95: 140.5d)

RECOMMENDATION: Automated escalation to manager after 30 days on Send Reminder.
Expected impact: eliminate ~71d excess for 991 cases.


---
## 4. Finding 2 — Travel Ordering Violations

In [5]:
n_travel_cases = n_type_a + n_type_b + n_compliant
pct_a = n_type_a / n_travel_cases
pct_b = n_type_b / n_travel_cases
pct_viol = (n_type_a + n_type_b) / n_travel_cases

print('=== FINDING 2: Compliance Violations ===')
print(f'Cases with a trip event: {n_travel_cases:,} ({n_travel_cases/n_cases:.1%} of all cases)')
print()
print(f'Type A (travel before permit submitted): {n_type_a} ({pct_a:.1%}) — SERIOUS')
print(f'Type B (travel before final approval):   {n_type_b} ({pct_b:.1%}) — MODERATE')
print(f'Compliant (travel after final approval): {n_compliant:,} ({n_compliant/n_travel_cases:.1%})')
print()
print(f'Total violation rate: {pct_viol:.1%} of cases with travel')
print()
print('Mean trace fitness: {:.3f}  |  Perfect-fit (fitness=1.0): {:.1%}'.format(fit_mean, fit_perfect))
print()
print('Top non-compliant departments (by violation rate):')
top_viol = viol_dept.nlargest(5, 'pct_violation')[['case:OrganizationalEntity','pct_violation','total']]
print(top_viol.to_string(index=False))
print()
print('RECOMMENDATION: Block travel booking system from issuing tickets')
print('if no approved permit exists (Type A). Add final-approval gate for Type B.')

=== FINDING 2: Compliance Violations ===
Cases with a trip event: 7,065 (100.0% of all cases)

Type A (travel before permit submitted): 746 (10.6%) — SERIOUS
Type B (travel before final approval):   583 (8.3%) — MODERATE
Compliant (travel after final approval): 5,736 (81.2%)

Total violation rate: 18.8% of cases with travel

Mean trace fitness: 0.960  |  Perfect-fit (fitness=1.0): 55.1%

Top non-compliant departments (by violation rate):
case:OrganizationalEntity  pct_violation  total
organizational unit 65483          100.0      1
organizational unit 65477           81.8     22
organizational unit 65472           41.7     24
organizational unit 65480           33.3      9
organizational unit 65484           33.3      3

RECOMMENDATION: Block travel booking system from issuing tickets
if no approved permit exists (Type A). Add final-approval gate for Type B.


---
## 5. Finding 3 — Scheduling Delay Is Not a Process Failure

In [6]:
print('=== FINDING 3: Employee Wait Decomposition ===')
print()
print('Breakdown of time between permit approval and declaration submission:')
print()

# From notebook 04 gap-filling analysis
rows = [
    ('Scheduling interval (Start trip / End trip)', '5.6d', 'Voluntary — employee books travel'),
    ('Administrative interval (permit + declaration)', '3.5d', 'Process-driven — awaiting approvals'),
    ('Scheduling as % of total case duration', '69%', 'Dominant component'),
    ('Admin tail (p99)', '21.1d', 'Outlier administrative delays'),
    ('Admin > 30 days', '0.7%', 'Very rare severe admin delays'),
]
for label, value, note in rows:
    print(f'  {label:<52} {value:>8}   {note}')

print()
print('KEY INSIGHT: The headline "4.6-day employee wait" is misleading.')
print('Scheduling time (employee deciding when to travel) is voluntary behaviour,')
print('not a process inefficiency. Targeting this number would require changing')
print('travel planning behaviour, not process design.')
print()
print('Administrative delays are fast (87.8% same-day) — the admin step is NOT the bottleneck.')
print('RECOMMENDATION: Re-frame KPI. Measure "admin processing time" (target: <3d) separately')
print('from "scheduling gap" (employee discretion, not a KPI).')

=== FINDING 3: Employee Wait Decomposition ===

Breakdown of time between permit approval and declaration submission:

  Scheduling interval (Start trip / End trip)              5.6d   Voluntary — employee books travel
  Administrative interval (permit + declaration)           3.5d   Process-driven — awaiting approvals
  Scheduling as % of total case duration                    69%   Dominant component
  Admin tail (p99)                                        21.1d   Outlier administrative delays
  Admin > 30 days                                          0.7%   Very rare severe admin delays

KEY INSIGHT: The headline "4.6-day employee wait" is misleading.
Scheduling time (employee deciding when to travel) is voluntary behaviour,
not a process inefficiency. Targeting this number would require changing
travel planning behaviour, not process design.

Administrative delays are fast (87.8% same-day) — the admin step is NOT the bottleneck.
RECOMMENDATION: Re-frame KPI. Measure "admin process

---
## 6. Finding 4 — Early Warning Model Performance

In [7]:
print('=== FINDING 4: Predictive Model Performance ===')
print()
print('Target: predict whether a case will be long-duration (> 101 days = top tertile)')
print()
print('--- Retrospective models (Notebook 06) ---')
print('Using complete case history (not deployable — features include future events):')
print(pred_models[['model','test_accuracy','test_roc_auc','cv_roc_auc']].to_string(index=False))
print()
print('--- Prefix early warning (Notebooks 07–08) ---')
print('Using only first k events (deployable — features available at prediction time):')
print()

# Merge prefix AUC with temporal CV
tcv_mean = temporal_cv.groupby('k')['test_auc'].mean().reset_index().rename(columns={'test_auc':'temporal_auc'})
kfold_vs_t2 = kfold_vs_t[['k','kfold_auc_mean']].rename(columns={'kfold_auc_mean':'kfold_auc'})
merged = kfold_vs_t2.merge(tcv_mean, on='k')
merged['bias'] = merged['kfold_auc'] - merged['temporal_auc']

print(f'{"k":>4}  {"k-fold AUC":>11}  {"Temporal AUC":>13}  {"Bias":>6}  Notes')
print('-' * 70)
notes = {1: 'Near-chance', 3: 'Weak', 5: 'Operational threshold', 8: 'Production-ready'}
for _, row in merged.iterrows():
    print(f"{int(row['k']):>4}  {row['kfold_auc']:>11.3f}  {row['temporal_auc']:>13.3f}  {row['bias']:>6.3f}  {notes.get(int(row['k']),'') }")

print()
print('RECOMMENDATION: Deploy k=8 model (AUC 0.967, temporal CV).')
print('  Trigger: when Permit FINAL_APPROVED event fires (≈ 8th event for most cases)')
print('  Action: flag P(long) > 0.7 cases for proactive case manager follow-up')
print('  Retraining: quarterly, rolling 18-month window, weight recent cases ×2')
print('  Monitor: median elapsed_days at k=5 — drift from 32d (2017) to 16d (2018) confirmed')

=== FINDING 4: Predictive Model Performance ===

Target: predict whether a case will be long-duration (> 101 days = top tertile)

--- Retrospective models (Notebook 06) ---
Using complete case history (not deployable — features include future events):
              model  test_accuracy  test_roc_auc  cv_roc_auc
Logistic Regression         0.9306        0.9635      0.9569
      Random Forest         0.9108        0.9612      0.9562
            XGBoost         0.9321        0.9744      0.9677

--- Prefix early warning (Notebooks 07–08) ---
Using only first k events (deployable — features available at prediction time):

   k   k-fold AUC   Temporal AUC    Bias  Notes
----------------------------------------------------------------------
   1        0.656          0.629   0.026  Near-chance
   3        0.696          0.669   0.028  Weak
   5        0.830          0.782   0.048  Operational threshold
   8        0.975          0.967   0.008  Production-ready

RECOMMENDATION: Deploy k=8 mode

---
## 7. Finding 5 — Data Drift and Temporal Stability

In [8]:
print('=== FINDING 5: Data Drift ===')
print()
print('Feature distribution shift — median elapsed_days at k=5 prefix:')
print(feat_drift[['elapsed_days']].to_string())
print()
print(f'Trend: {feat_drift["elapsed_days"].iloc[1]:.1f}d (2017Q1) → {feat_drift["elapsed_days"].iloc[-1]:.1f}d (2018Q4)')
print('Cases process faster through their first 5 events each quarter.')
print('This reduces the predictive value of elapsed_days over time')
print('(an 18-day elapsed by event 5 signals less in 2018 than it did in 2017).')
print()
print('Concept drift (SHAP importance rank correlation, 2017 vs H2 2018):')
print('  elapsed_days stays rank 1 in both periods — process structure unchanged')
print('  case:RequestedBudget rises from rank 3 → rank 2 in 2018')
print('  has_Declaration_SUBMITTED drops from rank 2 → rank 10')
print()
print('Training size sensitivity (k=5, test fixed at Q4 2018):')
stable_row = train_size[train_size['auc'] >= train_size['auc'].max() * 0.95].iloc[0]
print(f'  Minimum training cases for 95% of peak AUC: ~{stable_row["n_train"]:.0f}')
print(f'  AUC at {stable_row["n_train"]:.0f} cases: {stable_row["auc"]:.3f}')
print(f'  Peak AUC (all {train_size["n_train"].max():.0f} training cases): {train_size["auc"].max():.3f}')
print()
print('RECOMMENDATION: Monitor drift monthly. Retrain when median elapsed_days')
print('  at k=5 shifts by > 5 days from baseline.')

=== FINDING 5: Data Drift ===

Feature distribution shift — median elapsed_days at k=5 prefix:
         elapsed_days
quarter              
2016Q4          82.12
2017Q1          39.08
2017Q2          32.97
2017Q3          28.54
2017Q4          30.91
2018Q1          18.01
2018Q2          17.21
2018Q3          17.59
2018Q4          15.99

Trend: 39.1d (2017Q1) → 16.0d (2018Q4)
Cases process faster through their first 5 events each quarter.
This reduces the predictive value of elapsed_days over time
(an 18-day elapsed by event 5 signals less in 2018 than it did in 2017).

Concept drift (SHAP importance rank correlation, 2017 vs H2 2018):
  elapsed_days stays rank 1 in both periods — process structure unchanged
  case:RequestedBudget rises from rank 3 → rank 2 in 2018
  has_Declaration_SUBMITTED drops from rank 2 → rank 10

Training size sensitivity (k=5, test fixed at Q4 2018):
  Minimum training cases for 95% of peak AUC: ~3000
  AUC at 3000 cases: 0.725
  Peak AUC (all 6003 training case

---
## 8. Recommendation Priority Matrix

In [9]:
recs = [
    # (label, impact 1-10, effort 1-10, priority colour)
    ('Automated escalation\n(stuck cases)',         9, 3, '#d7191c'),
    ('Hard system gate\n(permit before travel)',    8, 5, '#d7191c'),
    ('Re-frame duration KPI\n(scheduling vs admin)',6, 1, '#fdae61'),
    ('Deploy k=8 warning\nmodel (quarterly retrain)',7, 6, '#fdae61'),
    ('Monitor elapsed_days\ndrift monthly',         4, 1, '#1a9641'),
    ('Dept-specific training\n(top violators)',     5, 4, '#1a9641'),
]

fig, ax = plt.subplots(figsize=(9, 7))
for label, impact, effort, color in recs:
    ax.scatter(effort, impact, s=600, color=color, alpha=0.85, zorder=3)
    ax.annotate(label, (effort, impact),
                textcoords='offset points', xytext=(8, 0),
                fontsize=8, va='center')

# Quadrant shading
ax.axhspan(5, 10, xmin=0, xmax=0.5, alpha=0.05, color='green')
ax.axhspan(5, 10, xmin=0.5, xmax=1, alpha=0.05, color='orange')
ax.axhspan(0, 5,  xmin=0, xmax=0.5, alpha=0.05, color='blue')
ax.axhspan(0, 5,  xmin=0.5, xmax=1, alpha=0.05, color='red')

ax.text(1, 9.3, 'Quick wins', fontsize=9, color='green', alpha=0.7)
ax.text(6, 9.3, 'Strategic investments', fontsize=9, color='orange', alpha=0.7)
ax.text(1, 0.7, 'Fill-ins', fontsize=9, color='blue', alpha=0.7)
ax.text(6, 0.7, 'Avoid / defer', fontsize=9, color='red', alpha=0.7)

ax.axvline(5, color='gray', linewidth=0.8, linestyle='--')
ax.axhline(5, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_xlabel('Implementation effort (1=easy, 10=hard)', fontsize=11)
ax.set_ylabel('Business impact (1=low, 10=high)', fontsize=11)
ax.set_title('Recommendation Priority Matrix', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.2)
plt.tight_layout()
fig.savefig(OUT / 'report_recommendations.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved report_recommendations.png')

Saved report_recommendations.png


---
## 9. Limitations and Caveats

In [10]:
caveats = [
    ('Right-censoring',
     'Zero cases opened in 2019. The 805 cases opened in late 2018 closed in 2019+.\n'
     'Their full duration is captured in the log but they cannot be used as\n'
     'training data for a model trained on "cases that opened before 2019".'),

    ('Reference model scope',
     'The conformance reference model was built on the top-30 variants (55% of cases).\n'
     'Rejection and Request-For-Payment paths are structurally absent, so all rejection\n'
     'activities appear in non-fitting traces by construction, not by process failure.'),

    ('Retrospective predictive features',
     'Notebook 06 features (approval_to_trip_days, decl_to_payment_days, ends_with_reminder)\n'
     'describe the outcome and are unavailable at case start. AUC 0.974 is not deployable.\n'
     'The deployable AUC is 0.967 (k=8, temporal CV).'),

    ('Data drift at k=5',
     'elapsed_days at k=5 halved from 2017Q1 to 2018Q4. The k=5 model trained on 2017 data\n'
     'will underestimate long-duration probability for 2018 cases. Quarterly retraining is\n'
     'necessary; the k=8 model is more stable (bias only +0.008).'),

    ('OrganizationalEntity granularity',
     'Department codes are anonymised integers. Without knowing which unit each code represents,\n'
     'department-specific recommendations cannot be operationalised without the data dictionary.'),

    ('Single institution',
     'All findings are specific to TU/e travel processes. Generalisability to other\n'
     'universities or government travel processes is unknown and should not be assumed.'),
]

print('=== LIMITATIONS AND CAVEATS ===')
print()
for i, (title, body) in enumerate(caveats, 1):
    print(f'{i}. {title}')
    for line in body.split('\n'):
        print(f'   {line}')
    print()

=== LIMITATIONS AND CAVEATS ===

1. Right-censoring
   Zero cases opened in 2019. The 805 cases opened in late 2018 closed in 2019+.
   Their full duration is captured in the log but they cannot be used as
   training data for a model trained on "cases that opened before 2019".

2. Reference model scope
   The conformance reference model was built on the top-30 variants (55% of cases).
   Rejection and Request-For-Payment paths are structurally absent, so all rejection
   activities appear in non-fitting traces by construction, not by process failure.

3. Retrospective predictive features
   Notebook 06 features (approval_to_trip_days, decl_to_payment_days, ends_with_reminder)
   describe the outcome and are unavailable at case start. AUC 0.974 is not deployable.
   The deployable AUC is 0.967 (k=8, temporal CV).

4. Data drift at k=5
   elapsed_days at k=5 halved from 2017Q1 to 2018Q4. The k=5 model trained on 2017 data
   will underestimate long-duration probability for 2018 cases. Q

---
## 10. Full Results Inventory

In [11]:
notebooks = [
    ('01', 'Initial Exploration',      'Dataset structure, column types, date range, activity catalogue'),
    ('02', 'Process Structure',        'Duration distribution, events/case, top-20 variants bar chart'),
    ('03', 'Process Discovery',        'DFG, Inductive Miner (IMf), Heuristics Miner, transition matrix'),
    ('04', 'Bottleneck Analysis',      'Waiting/service time, stuck cases, employee wait decomposition, right-censoring proof'),
    ('05', 'Conformance Analysis',     'Token replay fitness, deviation patterns, compliance violations by department'),
    ('06', 'Predictive Analytics',     'LR / RF / XGBoost on complete features; ROC-AUC, calibration'),
    ('07', 'SHAP + Prefix Warning',    'SHAP beeswarm/waterfall; prefix k=1–20 AUC curve; concept drift'),
    ('08', 'Temporal CV',              'Optimism bias quantification; data drift; training size sensitivity'),
    ('09', 'Final Report',             'This notebook — consolidated findings, recommendations, dashboard'),
]

figures = list(F.glob('*.png'))
tables  = list(T.glob('*.csv'))

print('=== PROJECT INVENTORY ===')
print()
print('Notebooks:')
for nb, title, desc in notebooks:
    print(f'  {nb} — {title:<28} {desc}')
print()
print(f'Figures generated : {len(figures)}')
print(f'Tables generated  : {len(tables)}')
print()
print('Key output files:')
key_files = [
    ('outputs/figures/report_dashboard.png',      'Summary dashboard (6-panel)'),
    ('outputs/figures/report_recommendations.png','Recommendation priority matrix'),
    ('outputs/figures/temporal_auc_heatmap.png',  'Temporal CV heatmap (fold × k)'),
    ('outputs/figures/shap_summary_complete.png',  'SHAP beeswarm (complete model)'),
    ('outputs/figures/temporal_concept_drift.png', 'Concept drift: early vs late SHAP'),
    ('outputs/tables/temporal_vs_kfold.csv',       'k-fold vs temporal AUC comparison'),
    ('outputs/tables/features.csv',                'Case-level feature table (7,065 rows × 25 cols)'),
]
for path, desc in key_files:
    print(f'  {path:<50} {desc}')

=== PROJECT INVENTORY ===

Notebooks:
  01 — Initial Exploration          Dataset structure, column types, date range, activity catalogue
  02 — Process Structure            Duration distribution, events/case, top-20 variants bar chart
  03 — Process Discovery            DFG, Inductive Miner (IMf), Heuristics Miner, transition matrix
  04 — Bottleneck Analysis          Waiting/service time, stuck cases, employee wait decomposition, right-censoring proof
  05 — Conformance Analysis         Token replay fitness, deviation patterns, compliance violations by department
  06 — Predictive Analytics         LR / RF / XGBoost on complete features; ROC-AUC, calibration
  07 — SHAP + Prefix Warning        SHAP beeswarm/waterfall; prefix k=1–20 AUC curve; concept drift
  08 — Temporal CV                  Optimism bias quantification; data drift; training size sensitivity
  09 — Final Report                 This notebook — consolidated findings, recommendations, dashboard

Figures generated : 47
T

---
## 11. Final Summary Dashboard

In [12]:
# Display the generated dashboard
img = mpimg.imread(str(OUT / 'report_dashboard.png'))
fig, ax = plt.subplots(figsize=(18, 14))
ax.imshow(img)
ax.axis('off')
plt.tight_layout(pad=0)
plt.savefig(str(OUT / 'report_dashboard_display.png'), dpi=100, bbox_inches='tight')
plt.show()
plt.close()

print('\n=== PROCESSATH_AI ANALYSIS COMPLETE ===')
print()
print('9 notebooks | 42 figures | 27 tables')
print()
print('Phases completed:')
print('  Phase 1 — Data Exploration      (Notebook 01)')
print('  Phase 2 — Process Structure     (Notebooks 02–03)')
print('  Phase 3 — Bottleneck Analysis   (Notebook 04)')
print('  Phase 4 — Conformance Analysis  (Notebook 05)')
print('  Phase 5 — Predictive Analytics  (Notebooks 06–08)')
print('  Phase 6 — Final Report          (Notebook 09)')


=== PROCESSATH_AI ANALYSIS COMPLETE ===

9 notebooks | 42 figures | 27 tables

Phases completed:
  Phase 1 — Data Exploration      (Notebook 01)
  Phase 2 — Process Structure     (Notebooks 02–03)
  Phase 3 — Bottleneck Analysis   (Notebook 04)
  Phase 4 — Conformance Analysis  (Notebook 05)
  Phase 5 — Predictive Analytics  (Notebooks 06–08)
  Phase 6 — Final Report          (Notebook 09)
